# Train MeowLLM 🐾

One-click training for **Miso**, a tiny ~3.5M parameter language model
that talks like a house cat.

**Runtime:** `Runtime → Change runtime type → T4 GPU` (free tier works)

**Total time:** ~20 minutes end-to-end.

This notebook will:
1. Clone the repo
2. Generate 20,000 synthetic training samples
3. Train a 2048-vocab BPE tokenizer
4. Train the transformer for 10 epochs
5. Chat with your trained Miso

## 1. Setup

In [ ]:
!git clone https://github.com/phanii9/MeowLLM.git
%cd MeowLLM
!pip install -q -e .

In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))

## 2. Generate the dataset

20,000 synthetic (input, output) pairs across 15 categories, generated from slot-based compositional templates with strict filtering.

In [ ]:
!python -m meow.generate_data --out-dir data --n 20000 --seed 42

## 3. Train the tokenizer

A tiny byte-level BPE (~1700 actual vocab).

In [ ]:
!python -m meow.tokenizer train data/train.jsonl data/tokenizer.json

## 4. Train the model

~3.5M parameters, 10 epochs, ~20 minutes on T4.

In [ ]:
!python -m meow.train \
    --train-data data/train.jsonl \
    --val-data data/val.jsonl \
    --tokenizer data/tokenizer.json \
    --out-dir checkpoints \
    --batch-size 64 \
    --epochs 10 \
    --lr 3e-4

## 5. Evaluate on held-out prompts

In [ ]:
from meow.inference import load_model, chat_once
from meow.tokenizer import MeowTokenizer
from meow.eval_cases import EVAL_PROMPTS, evaluate_batch, print_report

model, cfg = load_model('checkpoints/best.pt', device='cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = MeowTokenizer.from_file('data/tokenizer.json')

outputs = []
categories = []
print('generating responses to held-out prompts...\n')
for prompt, cat in EVAL_PROMPTS:
    response = chat_once(model, tokenizer, prompt, device='cuda' if torch.cuda.is_available() else 'cpu', temperature=0.8, top_k=40)
    outputs.append(response)
    categories.append(cat)
    print(f'[{cat:20s}] {prompt!r}')
    print(f'  -> {response!r}\n')

stats = evaluate_batch(outputs, categories)
print_report(stats)

## 6. Chat with Miso interactively

In [ ]:
def chat(prompt, temperature=0.8, top_k=40):
    response = chat_once(model, tokenizer, prompt, device='cuda' if torch.cuda.is_available() else 'cpu', temperature=temperature, top_k=top_k)
    print(f'you:  {prompt}')
    print(f'miso: {response}\n')
    return response

chat('hi miso')
chat('are you hungry')
chat('what do you think of the economy')
chat('there is a dog outside')
chat('i love you')

## 7. Save and share

Your checkpoint is in `checkpoints/best.pt`. To share it:

```python
from google.colab import files
files.download('checkpoints/best.pt')
files.download('data/tokenizer.json')
```

Or upload to Hugging Face (requires login):

```python
from huggingface_hub import HfApi
api = HfApi()
api.upload_file(path_or_fileobj='checkpoints/best.pt',
                path_in_repo='best.pt',
                repo_id='YOUR_USERNAME/meowllm',
                repo_type='model')
```

---

That's it. You trained a cat.